In [1]:
import pandas as pd

# ROOT
root_path = "C:/Users/jackm/Documents/GitHub/ms-capstone-TTS-natlang-styleprompts"

# ParaSpeechCaps

In [6]:
from datasets import load_dataset
import pandas as pd

# Load the entire dataset
# dataset = load_dataset("ajd12342/paraspeechcaps")

# Load specific splits of the dataset
# train_scaled = load_dataset("ajd12342/paraspeechcaps", split="train_scaled") #Only has EMilia files
train_base = load_dataset("ajd12342/paraspeechcaps", split="train_base")
holdout = load_dataset("ajd12342/paraspeechcaps", split="holdout")
dev = load_dataset("ajd12342/paraspeechcaps", split="dev")
test = load_dataset("ajd12342/paraspeechcaps", split="test")

dfs = [
    # train_scaled.to_pandas(),
    train_base.to_pandas(),
    holdout.to_pandas(),
    dev.to_pandas(),
    test.to_pandas(),
]

# merge into one dataframe
df = pd.concat(dfs, ignore_index=True)

# filter out VoxCeleb
df = df[df["source"] == "expresso"]
df = df[df["relative_audio_path"].str.contains("conversational_vad_segmented")]

KeyboardInterrupt: 

In [2]:
df.head()

,source,relative_audio_path,text_description,transcription,intrinsic_tags,situational_tags,basic_tags,all_tags,speakerid,name,duration,gender,accent,pitch,speaking_rate,noise,utterance_pitch_mean,snr,phonemes,tag_of_interest
17,expresso,audio_48khz/conversational_vad_segmented/ex04-...,"[ A female speaker's shrill, high-pitched voic...","Ugh, I mean, it's not my fault that you left ...","[booming, crisp, enunciated, loud, american, s...",[disgusted],"[female, high-pitched, slightly noisy environm...","[american, booming, crisp, disgusted, enunciat...",ex04,ex04,9.95,female,american,high-pitched,slow speed,slightly noisy environment,236.020462,39.719837,"ʌɡ, aɪ min, ɪt'ɛs nɑt maɪ fɔlt ðæt ju lɛft θɹ...",None
21,expresso,audio_48khz/conversational_vad_segmented/ex04-...,"[ A high-pitched, booming female voice speaks ...","Yeah, you know what? It doesn't matter. I thi...","[booming, crisp, enunciated, loud, american, s...",[disgusted],"[environment balanced in clarity, female, high...","[american, booming, crisp, disgusted, enunciat...",ex04,ex04,6.72,female,american,high-pitched,slow speed,environment balanced in clarity,224.929382,48.032288,"jæ, ju noʊ wʌt? ɪt 'ti mætɜ˞. aɪ θɪŋk wi kæn ...",None
25,expresso,audio_48khz/conversational_vad_segmented/ex04-...,[ A male American speaker's voice is high-pitc...,Is this a musical note right now? I want to s...,"[american, flowing, singsong]","[animated, laughing]","[high-pitched, male, measured speed, very clea...","[american, animated, flowing, high-pitched, la...",ex01,ex01,5.38,male,american,high-pitched,measured speed,very clean environment,234.016464,67.363716,ɪz ðɪs ʌ mjuzɪkʌl noʊt ɹaɪt naʊ? aɪ wɑnt tu s...,None
27,expresso,audio_48khz/conversational_vad_segmented/ex03-...,[ A male speaker with an American accent deliv...,You don't remember Mom and Dad's anniversary?...,"[authoritative, flowing, nasal, american, sing...",[calm],"[environment balanced in clarity, high-pitched...","[american, authoritative, calm, environment ba...",ex03,ex03,4.92,male,american,high-pitched,measured speed,environment balanced in clarity,188.732819,43.553577,ju dɑn'ti ɹɪmɛmbɜ˞ mɑm ʌnd dæd'ɛs ænʌvɜ˞sɜ˞i?...,None
30,expresso,audio_48khz/conversational_vad_segmented/ex01-...,[ A male speaker with an American accent deliv...,yeah we did that uh earlier yeah so we are we...,"[american, flowing, singsong]",[fast],"[high-pitched, male, measured speed, slightly ...","[american, fast, flowing, high-pitched, male, ...",ex01,ex01,11.78,male,american,high-pitched,measured speed,slightly clean environment,132.747528,51.455879,jæ wi dɪd ðæt ʌ ɜ˞liɜ˞ jæ soʊ wi ɑɹ wi ɑɹ sɑl...,None


In [10]:
# Duration in hours
total_hours = df["duration"].sum() / 3600
print(total_hours)

23.082813877314813


In [ ]:
# visualize tag distribution

import pandas as pd
import numpy as np

def is_populated(x):
    if x is None:
        return False
    if isinstance(x, float) and np.isnan(x):
        return False
    if isinstance(x, str):
        return x != 'None' and x.strip() != ''
    if isinstance(x, (list, tuple, np.ndarray)):
        return len(x) > 0
    return True  # fallback for other types

df['has_intrinsic'] = df['intrinsic_tags'].apply(is_populated)
df['has_situational'] = df['situational_tags'].apply(is_populated)

def categorize(row):
    if row['has_intrinsic'] and row['has_situational']:
        return 'both'
    elif row['has_intrinsic']:
        return 'intrinsic_only'
    elif row['has_situational']:
        return 'situational_only'
    else:
        return 'neither'

df['tag_category'] = df.apply(categorize, axis=1)

counts = (
    df.groupby(['source', 'tag_category'])
      .size()
      .unstack(fill_value=0)
)

import matplotlib.pyplot as plt

counts.plot(kind='bar')

plt.title('Tag Presence by Source')
plt.xlabel('Source')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Category')
plt.tight_layout()

plt.show()

# StyleTalk

In [3]:
styletalk_root = f"{root_path}/src/data/raw/styletalk"

In [ ]:
df_styletalk_eval_meta = pd.read_csv(
    f"{styletalk_root}/annotations/eval.csv",
    sep=",",            # delimiter
    encoding="utf-8",   # or "latin1"
    na_values=["NA", ""],
)

df_styletalk_eval_meta

In [ ]:
# Find total duration of StyleTalk: RESULTS = 7 hours

"""
Recursively sum the total duration of all WAV files in a directory.
Usage: python sum_wav_durations.py [directory]
"""

import sys
import wave
import os
from pathlib import Path
from tqdm import tqdm

# Sonnet 4.6: 

def get_wav_duration(filepath: Path) -> float:
    """Return duration of a WAV file in seconds, or None on error."""
    try:
        with wave.open(str(filepath), "rb") as f:
            frames = f.getnframes()
            rate = f.getframerate()
            return frames / rate
    except Exception as e:
        tqdm.write(f"[WARN] Could not read {filepath}: {e}")
        return None


def format_duration(seconds: float) -> str:
    """Format seconds into a human-readable string."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    parts = []
    if h:
        parts.append(f"{h}h")
    if m or h:
        parts.append(f"{m}m")
    parts.append(f"{s:.2f}s")
    return " ".join(parts)


def scan_directory(root: Path):
    total_seconds = 0.0
    file_count = 0
    error_count = 0

    wav_files = sorted(root.rglob("*.wav")) + sorted(root.rglob("*.WAV"))
    # Deduplicate (case-sensitive filesystems may return both)
    seen = set()
    unique_wavs = []
    for f in wav_files:
        if f not in seen:
            seen.add(f)
            unique_wavs.append(f)

    if not unique_wavs:
        print(f"No WAV files found under: {root}")
        return

    print(f"Scanning: {root}\n")

    with tqdm(unique_wavs, unit="file", dynamic_ncols=True) as pbar:
        for filepath in pbar:
            rel = filepath.relative_to(root)
            pbar.set_postfix_str(str(rel), refresh=True)

            duration = get_wav_duration(filepath)
            if duration is not None:
                total_seconds += duration
                file_count += 1
            else:
                error_count += 1

    print(f"\n{'─' * 60}")
    print(f"  Files found   : {file_count + error_count}")
    print(f"  Files read    : {file_count}")
    if error_count:
        print(f"  Errors        : {error_count}")
    print(f"  Total duration: {format_duration(total_seconds)}  ({total_seconds:.2f}s)")



directory = Path(f"{styletalk_root}/audio")

if not directory.exists():
    print(f"Error: directory not found: {directory}")
    sys.exit(1)
if not directory.is_dir():
    print(f"Error: not a directory: {directory}")
    sys.exit(1)

scan_directory(directory)

Scanning: C:\Users\jackm\Documents\GitHub\ms-capstone-TTS-natlang-styleprompts\src\data\raw\styletalk\audio



100%|██████████| 5392/5392 [00:35<00:00, 153.77file/s, work_98\r_2.wav]         



────────────────────────────────────────────────────────────
  Files found   : 5392
  Files read    : 5392
  Total duration: 7h 7m 54.19s  (25674.19s)
